# Count CSC faults & script failures per night over a long period

Trending companion to `count_night_failures.ipynb`. Same definitions, scaled to months.

**Scaling notes**
- Every topic here is a `logevent_*` (event-driven, sparse), *not* telemetry. A few months of `summaryState` / `Script.state` is small in row count; the cost is query count x per-query latency, plus the risk of wide-range InfluxDB queries timing out.
- Strategy: **one query per CSC per weekly chunk, sequential**. Bounds per-query latency and is timeout-resilient.
- Consecutive-state dedup runs on the *concatenated* per-CSC series, so chunk boundaries never fabricate or drop a FAULT transition.
- Binning: night-window only (sun <= -12 deg at Cerro Pachon), exactly like the MVP.

In [ ]:
import numpy as np
import pandas as pd
from astropy.time import Time
from astropy.coordinates import EarthLocation, AltAz, get_sun
import astropy.units as u
from lsst_efd_client import EfdClient

## Configuration

In [ ]:
EFD_NAME   = 'usdf_efd'        # or 'summit_efd'
START_DATE = '2026-01-01'      # first observing date (inclusive)
END_DATE   = '2026-05-15'      # last  observing date (inclusive)

# Correlation window around each script failure (seconds).
CORRELATION_PRE_S  = 35.0      # generous: dome-style delayed failures
CORRELATION_POST_S = 5.0       # small slack for timestamp jitter

CHUNK_DAYS = 7                 # weekly EFD query windows, run sequentially

USE_GROUPS = {
    'maintel':         True,
    'camera':          True,
    'ess':             True,
    'dome_subsystems': False,  # stub; see MVP notebook
}

## CSC groups

Same list as the MVP. Format: `(csc_name, sal_index_or_None)`.

In [ ]:
CSC_GROUPS = {
    'maintel': [
        ('MTMount',          None),
        ('MTPtg',            None),
        ('MTAOS',            None),
        ('MTRotator',        None),
        ('MTHexapod',        1),     # camera hexapod
        ('MTHexapod',        2),     # M2 hexapod
        ('MTM1M3',           None),
        ('MTM1M3TS',         None),
        ('MTM2',             None),
        ('MTDome',           None),
        ('MTDomeTrajectory', None),
    ],
    'camera': [
        ('MTCamera',         None),  # LSSTCam - confirm CSC name in your environment
        ('MTHeaderService',  None),
        ('MTOODS',           None),
    ],
    'ess': [('ESS', idx) for idx in [1, 101, 102, 103, 104, 105, 106, 107, 108,
                                     201, 202, 203, 204, 205, 301]],
}

## Night windows for the whole period

One continuous 60-s grid from noon-UTC(START) to noon-UTC(END+1), a single `get_sun` call, then split into contiguous `sun <= -12 deg` runs. Run *k* is the night of `START + k days`. 60-s resolution moves a twilight bound by <= 1 min, which never reassigns a mid-night fault.

In [ ]:
RUBIN_LOC = EarthLocation.from_geodetic(
    lon=-70.749417*u.deg, lat=-30.244639*u.deg, height=2663*u.m,
)

def build_nights(start_date, end_date, alt_limit_deg=-12.0, step_s=60):
    t0 = Time(f'{start_date}T12:00:00', scale='utc')
    t1 = Time(f'{end_date}T12:00:00', scale='utc') + 1*u.day
    n  = int(round((t1 - t0).to(u.s).value / step_s)) + 1
    times = t0 + np.arange(n) * step_s * u.s
    alt = get_sun(times).transform_to(AltAz(obstime=times, location=RUBIN_LOC)).alt.deg
    below = alt <= alt_limit_deg
    d = np.diff(below.astype(int))
    starts = np.where(d == 1)[0] + 1
    ends   = np.where(d == -1)[0]
    if below[0]:
        starts = np.r_[0, starts]
    if below[-1]:
        ends = np.r_[ends, len(below) - 1]
    dates = pd.date_range(start_date, end_date, freq='D')
    if len(starts) != len(dates):
        print(f'WARNING: {len(starts)} night segments for {len(dates)} dates; '
              f'observing-date alignment may be off.')
    rows = []
    for k, (s, e) in enumerate(zip(starts, ends)):
        rows.append({
            'observing_date': dates[k] if k < len(dates) else pd.NaT,
            't_start': pd.Timestamp(times[s].to_datetime(), tz='UTC'),
            't_end':   pd.Timestamp(times[e].to_datetime(), tz='UTC'),
        })
    return pd.DataFrame(rows).set_index('observing_date')

nights = build_nights(START_DATE, END_DATE)
print(f'{len(nights)} nights: {nights.index.min().date()} -> {nights.index.max().date()}  '
      f'(mean {(nights.t_end - nights.t_start).mean()})')
nights.head()

## EFD client

In [ ]:
client = EfdClient(EFD_NAME)

## Weekly query chunks

Queries span only `[first night start, last night end]`. Daytime gaps between nights are still queried (a single contiguous chunk is cheaper than carving each night out), but non-night events are dropped at the binning step.

In [ ]:
GLOBAL_T0 = nights['t_start'].min()
GLOBAL_T1 = nights['t_end'].max()

def make_chunks(t0, t1, days):
    edges = list(pd.date_range(t0, t1, freq=f'{days}D'))
    if not edges or edges[0] > t0:
        edges = [t0] + edges
    if edges[-1] < t1:
        edges.append(t1)
    return list(zip(edges[:-1], edges[1:]))

CHUNKS  = make_chunks(GLOBAL_T0, GLOBAL_T1, CHUNK_DAYS)
to_time = lambda ts: Time(pd.Timestamp(ts).to_pydatetime(), scale='utc')
print(f'{len(CHUNKS)} chunks of <= {CHUNK_DAYS}d  |  {GLOBAL_T0}  ->  {GLOBAL_T1}')

## Per-night binning helper

In [ ]:
_IV = pd.IntervalIndex.from_arrays(nights['t_start'], nights['t_end'], closed='both')

def assign_night(df):
    # Map each UTC-indexed row to its observing date; drop rows outside any night.
    if df.empty:
        return df.assign(night=pd.Series(dtype='datetime64[ns]'))
    loc = _IV.get_indexer(df.index)
    night = np.where(loc >= 0, nights.index.values[loc.clip(0)], np.datetime64('NaT'))
    df = df.assign(night=pd.to_datetime(night))
    return df[df['night'].notna()].copy()

## CSC faults (transitions into `summaryState == FAULT`)

In [ ]:
FAULT_STATE = 3  # SAL State: DISABLED=1, ENABLED=2, FAULT=3, OFFLINE=4, STANDBY=5

async def fetch_summary_state(csc, idx):
    topic = f'lsst.sal.{csc}.logevent_summaryState'
    kw = {'index': idx} if idx is not None else {}
    parts = []
    for c0, c1 in CHUNKS:
        try:
            d = await client.select_time_series(topic, ['summaryState'],
                                                to_time(c0), to_time(c1), **kw)
        except Exception as e:
            print(f'  [skip] {csc}:{idx} {pd.Timestamp(c0).date()} ({type(e).__name__})')
            continue
        if not d.empty:
            parts.append(d)
    if not parts:
        return pd.DataFrame()
    df = pd.concat(parts).sort_index()
    df = df[~df.index.duplicated(keep='first')]                # drop chunk-edge repeats
    df = df[df['summaryState'] != df['summaryState'].shift()]  # keep true transitions
    df = df[df['summaryState'] == FAULT_STATE].copy()
    df['csc'] = f'{csc}:{idx}' if idx is not None else csc
    return df

async def collect_csc_faults():
    selected = []
    for grp, on in USE_GROUPS.items():
        if grp == 'dome_subsystems' or not on:
            continue
        selected.extend(CSC_GROUPS.get(grp, []))
    parts = []
    for name, idx in selected:
        df = await fetch_summary_state(name, idx)
        print(f'  {name}:{idx}  faults={0 if df.empty else len(df)}')
        if not df.empty:
            parts.append(assign_night(df))
    cols = ['summaryState', 'csc', 'night']
    return pd.concat(parts).sort_index() if parts else pd.DataFrame(columns=cols)

csc_faults = await collect_csc_faults()
print(f'CSC fault transitions (in night windows): {len(csc_faults)}')
csc_faults['csc'].value_counts() if not csc_faults.empty else None

## Script failures (`Script.logevent_state` terminal failure states)

In [ ]:
SCRIPT_FAIL_STATES = (10, 11)  # ScriptState: FAILED=10, CONFIGURE_FAILED=11

async def get_script_failures():
    parts = []
    for c0, c1 in CHUNKS:
        try:
            d = await client.select_time_series(
                'lsst.sal.Script.logevent_state', ['state', 'ScriptID'],
                to_time(c0), to_time(c1))
        except Exception as e:
            print(f'  [skip] Script {pd.Timestamp(c0).date()} ({type(e).__name__})')
            continue
        if not d.empty:
            parts.append(d)
    if not parts:
        return pd.DataFrame(columns=['state', 'ScriptID', 'night'])
    df = pd.concat(parts).sort_index()
    df = df[~df.index.duplicated(keep='first')]
    df = df[df['state'].isin(SCRIPT_FAIL_STATES)]
    df = assign_night(df)
    # ScriptID recycles over months -> dedup per night, not globally.
    # NOTE: the MVP saw ScriptID == None; if it is unusable here we keep every
    # terminal-state row (one per failed execution after the state filter).
    if df['ScriptID'].notna().any():
        df = df.drop_duplicates(subset=['night', 'ScriptID'], keep='first')
    return df

script_failures = await get_script_failures()
print(f'Script failures (in night windows): {len(script_failures)}')
script_failures.head()

## Dome subsystem faults (stub)

Same open item as the MVP: MTDome subsystems can degrade while `MTDome.summaryState` stays ENABLED. Returns empty until the signal is chosen.

In [ ]:
async def get_dome_subsystem_faults():
    # TODO: implement once the chosen MTDome subsystem signal is confirmed.
    return pd.DataFrame(columns=['csc', 'night'])

dome_subs = (await get_dome_subsystem_faults()
             if USE_GROUPS['dome_subsystems']
             else pd.DataFrame(columns=['csc', 'night']))
print(f'Dome subsystem faults: {len(dome_subs)}')

## Correlate & aggregate per night

Attribution is time-local (35 s), so it is done within each night: a script failure within `[t - pre, t + post]` of any CSC fault is attributed to that fault and not counted as a standalone event. `unique_total = csc_faults + standalone_script_failures` (+ dome subs once wired).

In [ ]:
def attribute(faults_df, scripts_df, pre_s, post_s):
    if scripts_df.empty:
        return scripts_df.assign(attributed=pd.Series(dtype=bool))
    if faults_df.empty:
        return scripts_df.assign(attributed=False)
    f_t  = faults_df.index.values.astype('datetime64[ns]')
    pre  = np.timedelta64(int(pre_s  * 1e9), 'ns')
    post = np.timedelta64(int(post_s * 1e9), 'ns')
    flags = [((f_t >= t - pre) & (f_t <= t + post)).any()
             for t in scripts_df.index.values.astype('datetime64[ns]')]
    return scripts_df.assign(attributed=flags)

all_faults = (pd.concat([csc_faults, dome_subs])
              if not dome_subs.empty else csc_faults)

marked = []
for night, s in script_failures.groupby('night'):
    f = all_faults[all_faults['night'] == night]
    marked.append(attribute(f, s, CORRELATION_PRE_S, CORRELATION_POST_S))
scripts_marked = (pd.concat(marked) if marked
                  else script_failures.assign(attributed=pd.Series(dtype=bool)))

per_night = pd.DataFrame(index=nights.index)
per_night['csc_faults'] = all_faults.groupby('night').size()
per_night['script_total']      = 0
per_night['script_attributed'] = 0
if not scripts_marked.empty:
    g = scripts_marked.groupby('night')['attributed']
    per_night['script_total']      = g.size()
    per_night['script_attributed'] = g.sum()
per_night = per_night.fillna(0).astype(int)
per_night['script_standalone'] = per_night['script_total'] - per_night['script_attributed']
per_night['unique_total'] = per_night['csc_faults'] + per_night['script_standalone']
per_night['roll7'] = per_night['unique_total'].rolling(7, min_periods=1).mean()
per_night.tail()

## Summary

In [ ]:
n_csc = len(all_faults)
n_scr = len(scripts_marked)
n_att = int(scripts_marked['attributed'].sum()) if n_scr else 0
n_std = n_scr - n_att
n_tot = int(per_night['unique_total'].sum())
bar = '=' * 60

print(bar)
print(f' Period: {START_DATE} -> {END_DATE}   ({len(nights)} nights)')
print(f' Correlation: script within [-{CORRELATION_PRE_S:.0f}s, +{CORRELATION_POST_S:.0f}s] of a CSC fault')
print(bar)
print(f' CSC fault transitions:        {n_csc}')
print(f' Dome subsystem faults:        {len(dome_subs)}')
print(f' Script failures (total):      {n_scr}')
print(f'   attributed to CSC fault:    {n_att}')
print(f'   standalone:                 {n_std}')
print(bar)
print(f' TOTAL UNIQUE FAILURE EVENTS:  {n_tot}')
print(bar)
if not all_faults.empty:
    print()
    print('Top CSC offenders:')
    print(all_faults['csc'].value_counts().head(15).to_string())

## Trend plots (Bokeh, interactive)

In [ ]:
from bokeh.plotting import figure, show
from bokeh.io import output_notebook
from bokeh.models import ColumnDataSource, HoverTool
from bokeh.palettes import Category10
output_notebook()

src = ColumnDataSource(per_night.reset_index())
p = figure(x_axis_type='datetime', height=380, width=1000,
           title='Failure events per night',
           x_axis_label='Observing date', y_axis_label='Events')
p.vbar_stack(['csc_faults', 'script_standalone'], x='observing_date', width=6.5e7,
             color=list(Category10[3][:2]), source=src,
             legend_label=['CSC faults', 'Standalone script failures'])
p.line('observing_date', 'roll7', source=src, color='black', line_width=2,
        legend_label='7-night mean (total)')
p.add_tools(HoverTool(
    tooltips=[('night', '@observing_date{%F}'),
              ('CSC faults', '@csc_faults'),
              ('script standalone', '@script_standalone'),
              ('total', '@unique_total')],
    formatters={'@observing_date': 'datetime'}))
p.legend.location = 'top_left'
p.legend.click_policy = 'hide'
show(p)

### Which CSCs drive the faults (top offenders over time)

In [ ]:
TOP_N = 8
if not all_faults.empty:
    pivot = (all_faults.assign(n=1)
             .pivot_table(index='night', columns='csc', values='n',
                          aggfunc='sum', fill_value=0)
             .reindex(nights.index, fill_value=0))
    top = pivot.sum().sort_values(ascending=False).head(TOP_N).index.tolist()
    pv  = pivot[top].rename_axis('night').reset_index()
    pal = (list(Category10[10]) * (len(top) // 10 + 1))[:len(top)]
    p2 = figure(x_axis_type='datetime', height=400, width=1000,
                title=f'CSC fault transitions per night (top {TOP_N} CSCs)',
                x_axis_label='Observing date', y_axis_label='Fault transitions')
    p2.vbar_stack(top, x='night', width=6.5e7, color=pal,
                  source=ColumnDataSource(pv), legend_label=top)
    p2.legend.location = 'top_left'
    p2.legend.click_policy = 'hide'
    show(p2)
else:
    print('No CSC faults to break down.')

## Publication plot (matplotlib)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 4))
x = per_night.index
ax.bar(x, per_night['csc_faults'], label='CSC faults')
ax.bar(x, per_night['script_standalone'], bottom=per_night['csc_faults'],
       label='Standalone script failures')
ax.plot(x, per_night['roll7'], color='black', lw=1.5, label='7-night mean')
ax.set_xlabel('Observing date')
ax.set_ylabel('Failure events')
ax.set_title(f'Failure events per night   {START_DATE} -> {END_DATE}')
ax.legend()
fig.autofmt_xdate()
fig.tight_layout()